# VCell-AI Knowledge Base Seeding

One-time (or occasional) script to populate the Qdrant knowledge base backing the RAG pipeline at `vcell-ai-dev.cam.uchc.edu`.

**Sources scraped:**
- `vcell.org/webstart/VCell_Tutorials/VCell_Help/` (HTML tutorial pages)
- `vcell.org/webstart/VCell_Tutorials/` (top-level PDF tutorials)
- `vcell.org/webstart/VCell_Tutorials/7.7/` (v7.7 PDF tutorials)

**Flow:** scrape source directories → download HTML+PDF locally → delete existing entries for each filename in the KB → upload to backend, which chunks, embeds via OpenAI `text-embedding-3-small`, and upserts into Qdrant.

Run cells top to bottom.

In [1]:
import os
from urllib.parse import urlparse, unquote

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()

# --- Config ---
BASE = "https://vcell-ai-dev.cam.uchc.edu/api"
# For local testing against docker-compose, use: BASE = "http://127.0.0.1:8000"

TOKEN = os.environ.get("VCELL_KB_ADMIN_TOKEN")
if not TOKEN:
    raise RuntimeError(
        "Missing VCELL_KB_ADMIN_TOKEN in environment. "
        "Add it to a .env file in this notebook's directory."
    )

HEADERS = {"Authorization": f"Bearer {TOKEN}"}

DOWNLOADS_DIR = os.path.join(os.getcwd(), "Downloads")
os.makedirs(DOWNLOADS_DIR, exist_ok=True)

# filename → original source URL; used to attach source_url on upload
source_url_by_filename: dict[str, str] = {}

print(f"Backend:        {BASE}")
print(f"Downloads dir:  {DOWNLOADS_DIR}")

Backend:        https://vcell-ai-dev.cam.uchc.edu/api
Downloads dir:  /Users/kartikdeshpande/Desktop/vcell-repos/VCell-AI/backend/Downloads


In [3]:
# Confirm backend is reachable and the token is accepted as admin.
r = requests.get(f"{BASE}/kb/files", headers=HEADERS, timeout=30)
print(f"GET /kb/documents → {r.status_code}")

if r.status_code == 401 or r.status_code == 403:
    raise RuntimeError("Auth failed — check VCELL_KB_ADMIN_TOKEN and role.")
if r.status_code != 200:
    raise RuntimeError(f"Backend not healthy: {r.status_code} {r.text[:300]}")

try:
    existing = r.json()
    print(f"Existing entries in KB: {len(existing) if isinstance(existing, list) else 'n/a'}")
except Exception:
    print("(response was not JSON)")

GET /kb/documents → 200
Existing entries in KB: n/a


---
## Step 1 — Scrape HTML tutorial pages

In [5]:
HTML_INDEX_URL = "https://vcell.org/webstart/VCell_Tutorials/VCell_Help/index.html"
HTML_BASE = "https://vcell.org/webstart/VCell_Tutorials/VCell_Help/"


def sanitize_filename(url_path: str) -> str:
    """Turn a URL path into a safe flat filename ending in .txt."""
    return url_path.strip("/").replace("/", "__") + ".txt"


r = requests.get(HTML_INDEX_URL, timeout=30)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

hrefs = [a["href"] for a in soup.find_all("a", href=True)]
html_links = [h if h.startswith("http") else HTML_BASE + h for h in hrefs]

print(f"Found {len(html_links)} HTML links from the index page")

Found 142 HTML links from the index page


In [6]:
saved = 0
for url in html_links:
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"  SKIP {url} (HTTP {r.status_code})")
            continue

        page = BeautifulSoup(r.text, "html.parser")
        for tag in page(["script", "style", "head", "title", "meta"]):
            tag.extract()
        text = page.get_text(separator="\n", strip=True)

        filename = sanitize_filename(url)
        filepath = os.path.join(DOWNLOADS_DIR, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(text)

        source_url_by_filename[filename] = url
        saved += 1
    except Exception as e:
        print(f"  ERROR {url}: {e}")

print(f"Saved {saved} HTML pages as text files")

Saved 142 HTML pages as text files


---
## Step 2 — Scrape and download PDF tutorials

In [7]:
PDF_DIRS = [
    "https://vcell.org/webstart/VCell_Tutorials/",
    "https://vcell.org/webstart/VCell_Tutorials/7.7/",
]

pdf_links: list[str] = []
for base in PDF_DIRS:
    r = requests.get(base, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            full = href if href.startswith("http") else base + href
            pdf_links.append(full)

# dedupe, preserve order
pdf_links = list(dict.fromkeys(pdf_links))
print(f"Found {len(pdf_links)} PDF files across both directories")

Found 22 PDF files across both directories


In [8]:
for url in pdf_links:
    filename = unquote(os.path.basename(urlparse(url).path))
    filepath = os.path.join(DOWNLOADS_DIR, filename)
    try:
        r = requests.get(url, timeout=180)
        r.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(r.content)
        source_url_by_filename[filename] = url
        print(f"  {filename}  ({len(r.content) / 1e6:.1f} MB)")
    except Exception as e:
        print(f"  ERROR {filename}: {e}")

pdf_count = sum(1 for fn in source_url_by_filename if fn.lower().endswith(".pdf"))
print(f"\nDownloaded {pdf_count} PDFs")

  FRAPBinding_7.0.pdf  (8.3 MB)
  FRAPBinding_7.2.pdf  (17.8 MB)
  MovingBoundaries.pdf  (9.9 MB)
  MultiAppTransport_7.0.pdf  (14.4 MB)
  MultiAppTransport_7.2.pdf  (16.6 MB)
  PHGFP_7.0.pdf  (5.8 MB)
  PHGFP_7.2.pdf  (9.9 MB)
  SimpleFRAP_7.0.pdf  (3.9 MB)
  SimpleFRAP_7.2.pdf  (8.6 MB)
  SingleCompartmentRuleBased.pdf  (1.3 MB)
  SpatialRuleBasedGuide.pdf  (0.7 MB)
  Tutorial06_PathwayCommons_6.0.pdf  (4.8 MB)
  VCell6.1_Rule-Based_Ran_Transport_Tutorial.pdf  (10.3 MB)
  VCell6.1_Rule-Based_Tutorial.pdf  (8.6 MB)
  VCell_Quickstart_6.pdf  (0.9 MB)
  VCell_Quickstart_7_Biomodel.pdf  (0.7 MB)
  VCell Quick Guide_ Image Based Geometry 7.7.pdf  (1.7 MB)
  VCell Tutorial_ Constructive Solid Geometry_ Dendritic Spine 7.7.pdf  (2.3 MB)
  VCell Tutorial_ Image-Based Geometry 7.7.pdf  (5.2 MB)
  VCell Tutorial_ Model Physiology 7.7.pdf  (2.5 MB)
  VCell Tutorial_ Rule-Based EGFR 7.7.pdf  (8.2 MB)
  VCell Tutorial_ Rule-Based Ran Transport 7.7.pdf  (10.6 MB)

Downloaded 22 PDFs


---
## Step 3 — Upload to Knowledge Base

Each file is (a) deleted from the KB first (best-effort, ignoring 404s
so the first-ever seed works too), then (b) re-uploaded. This keeps the
notebook idempotent — safe to re-run after a partial failure or when
sources are refreshed.

In [9]:
pdf_endpoint = f"{BASE}/kb/upload-pdf"
text_endpoint = f"{BASE}/kb/upload-text"


def delete_existing(filename: str) -> None:
    """Best-effort delete; ignore missing files and network hiccups."""
    try:
        requests.delete(
            f"{BASE}/kb/files/{filename}",
            headers=HEADERS,
            timeout=30,
        )
    except Exception:
        pass


uploaded, failed, skipped = 0, 0, 0

for filename in sorted(os.listdir(DOWNLOADS_DIR)):
    filepath = os.path.join(DOWNLOADS_DIR, filename)
    if not os.path.isfile(filepath):
        continue

    ext = os.path.splitext(filename)[1].lower()
    source_url = source_url_by_filename.get(filename, "")

    if ext == ".pdf":
        endpoint, mime = pdf_endpoint, "application/pdf"
    elif ext == ".txt":
        endpoint, mime = text_endpoint, "text/plain"
    else:
        skipped += 1
        print(f"  SKIP {filename} (unsupported extension: {ext})")
        continue

    delete_existing(filename)

    try:
        with open(filepath, "rb") as f:
            files = {"file": (filename, f, mime)}
            data = {"source_url": source_url}
            r = requests.post(
                endpoint,
                files=files,
                data=data,
                headers=HEADERS,
                timeout=600,
            )

        if r.status_code == 200:
            uploaded += 1
            print(f"  OK    {filename}")
        else:
            failed += 1
            print(f"  FAIL  {filename} → HTTP {r.status_code}: {r.text[:200]}")
    except Exception as e:
        failed += 1
        print(f"  ERROR {filename}: {e}")

print(f"\nSummary — uploaded: {uploaded}, failed: {failed}, skipped: {skipped}")

  OK    FRAPBinding_7.0.pdf
  OK    FRAPBinding_7.2.pdf
  OK    MovingBoundaries.pdf
  OK    MultiAppTransport_7.0.pdf
  OK    MultiAppTransport_7.2.pdf
  OK    PHGFP_7.0.pdf
  OK    PHGFP_7.2.pdf
  OK    SimpleFRAP_7.0.pdf
  OK    SimpleFRAP_7.2.pdf
  OK    SingleCompartmentRuleBased.pdf
  OK    SpatialRuleBasedGuide.pdf
  OK    Tutorial06_PathwayCommons_6.0.pdf
  OK    VCell Quick Guide_ Image Based Geometry 7.7.pdf
  OK    VCell Tutorial_ Constructive Solid Geometry_ Dendritic Spine 7.7.pdf
  OK    VCell Tutorial_ Image-Based Geometry 7.7.pdf
  OK    VCell Tutorial_ Model Physiology 7.7.pdf
  OK    VCell Tutorial_ Rule-Based EGFR 7.7.pdf
  OK    VCell Tutorial_ Rule-Based Ran Transport 7.7.pdf
  OK    VCell6.1_Rule-Based_Ran_Transport_Tutorial.pdf
  OK    VCell6.1_Rule-Based_Tutorial.pdf
  OK    VCell_Quickstart_6.pdf
  OK    VCell_Quickstart_7_Biomodel.pdf
  OK    https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__Auth0__Auth0ExistingUserAccountLinking.html.txt
  O

---
## Step 4 — Verify

In [11]:
r = requests.get(f"{BASE}/kb/files", headers=HEADERS, timeout=30)
if r.status_code == 200:
    try:
        docs = r.json()
        if isinstance(docs, list):
            print(f"Knowledge base now contains {len(docs)} entries.")
            for d in docs[:10]:
                print(f"  - {d}")
            if len(docs) > 10:
                print(f"  ... and {len(docs) - 10} more")
        else:
            print(docs)
    except Exception:
        print(r.text[:500])
else:
    print(f"Verify failed: {r.status_code} {r.text[:300]}")

{'status': 'success', 'files': ['https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_1__Introduction__TopMenu.html.txt', 'https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_4__ParametersAndFunctions__ParametersAndFunctions.html.txt', 'https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_10__RuleBasedChapter__NFSimApp.html.txt', 'https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_8__PropertiesPanes__PP_Problems.html.txt', 'https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_1__Introduction__Open.html.txt', 'https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_8__PropertiesPanes__PP_ApplicationProps.html.txt', 'https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_3__BioModelApplications__Specifications__SpecificationsOverview.html.txt', 'https:____vcell.org__webstart__VCell_Tutorials__VCell_Help__topics__ch_8__PropertiesPanes__PropertiesPaneOverview.html.txt